# PLTR exploratory analysis

Explore daily returns, market sensitivity, and drawdown over the collected

price window: May 1, 2023, through April 30, 2025.

In [1]:
import numpy as np, pandas as pd, json, sys, yfinance as yf, plotly.express as px, scipy.stats as sc, requests
start, end = "2023-05-01", "2025-05-01"
purchase_date = "2024-02-01"

In [2]:
PLTR = pd.read_csv("../data/pltr_prices.csv", index_col=0, parse_dates=True).sort_index()

print(PLTR.index.min(), PLTR.index.max())
display(PLTR.head())
display(PLTR.isna().sum())

2023-05-01 00:00:00 2025-04-30 00:00:00


,Close,High,Low,Open,Volume
Date,,,,,
2023-05-01,7.78,7.82,7.66,7.75,19134200
2023-05-02,7.59,7.91,7.43,7.72,38008000
2023-05-03,7.56,7.68,7.45,7.62,24322400
2023-05-04,7.38,7.52,7.28,7.47,29989300
2023-05-05,7.41,7.55,7.32,7.44,45848800


Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [3]:
returns = PLTR["Close"].pct_change(fill_method=None).dropna() * 100
display(returns.describe())

count    501.000000
mean       0.641696
std        4.504065
min      -15.113049
25%       -1.688989
50%        0.258919
75%        2.277533
max       30.801446
Name: Close, dtype: float64

## Historical daily-return distribution

The sample mean estimates average daily return over this historical window.

The proportions within one and two standard deviations are measured directly
from the data.

In [4]:
μ, std = returns.mean(), returns.std()
fig = px.histogram(x=returns, nbins=50, width=900, height=300, labels={"x": "Daily return (%)"}).update_layout(margin=dict(l=5, r=00, t=20, b=10))
fig.add_vline(
    x=μ, line_color="red", line_width=3,
    annotation_text=f"μ = {μ:.2f}%",
    annotation_position="top",
    annotation_font_color="red"
)
fig.add_annotation(
    x=0.65, y=0.75, xref="paper", yref="paper",
    text=f"Returns within mean ±σ: {returns.between(μ - std, μ + std).mean():.1%}",
    showarrow=False, xanchor="left", yanchor="top"
)
fig.add_annotation(
    x=0.65, y=0.55, xref="paper", yref="paper",
    text=f"Returns within mean ±2σ: {returns.between(μ - 2*std, μ + 2*std).mean():.1%}",
    showarrow=False, xanchor="left", yanchor="top"
)
for n in (1, 2):
    for s in (+std, -std):
        fig.add_vline(
            x=μ + n*s, line_color="red", line_width=3,
            annotation_text=f"{'+' if s > 0 else '-'}{n}σ = {(μ + n*s):.1f}%",
            annotation_position=f"top {"right" if s > 0 else "left"}",
            annotation_font_color="red")
fig.show()
print(f"{returns.between(μ - std, μ + std).mean():.1%}")

80.4%


## Compared to S&P500 daily returns

In [5]:
market_returns = yf.download("SPY", start=start, end=end, auto_adjust=False, multi_level_index=False)["Close"].pct_change(fill_method=None).dropna() * 100
μ, std = market_returns.mean(), market_returns.std()
fig = px.histogram(x=market_returns, nbins=50, width=900, height=300, labels={"x": "Daily return (%)"}).update_layout(margin=dict(l=5, r=00, t=20, b=10))
fig.add_annotation(
    x=0.65, y=0.75, xref="paper", yref="paper",
    text=f"Returns within mean ±σ: {market_returns.between(μ - std, μ + std).mean():.1%}",
    showarrow=False, xanchor="left", yanchor="top"
)
fig.add_annotation(
    x=0.65, y=0.55, xref="paper", yref="paper",
    text=f"Returns within mean ±2σ: {market_returns.between(μ - 2*std, μ + 2*std).mean():.1%}",
    showarrow=False, xanchor="left", yanchor="top"
)
fig.add_vline(
    x=μ, line_color="red", line_width=3,
    annotation_text=f"μ = {μ:.2f}%",
    annotation_position="top",
    annotation_font_color="red"
)
for n in (1, 2):
    for s in (+std, -std):
        fig.add_vline(
            x=μ + n*s, line_color="red", line_width=3,
            annotation_text=f"{'+' if s > 0 else '-'}{n}σ",
            annotation_position=f"top {"right" if s > 0 else "left"}",
            annotation_font_color="red")
fig.show()

[*********************100%***********************]  1 of 1 completed


## Correlation and market sensitivity

Returns are aligned by trading date before comparison. Correlation measures
their linear association.

The dashed `y = x` line is an equal-return reference,
not a fitted regression.

In [6]:
comparison = pd.concat({"PLTR": returns, "Market": market_returns}, axis=1).dropna()

display(comparison.corr())
fig = px.scatter(comparison, x="Market", y="PLTR", width=600, height=300)
fig.add_annotation(
    x=0.05, y=0.62, xref="paper", yref="paper",
    text=f"Corr ~ {comparison.corr()["PLTR"].loc["Market"]:.2f}",
    showarrow=False, xanchor="left", yanchor="top"
)
lo, hi = comparison["Market"].min(), comparison["Market"].max()
fig.add_shape(type="line", x0=lo, y0=lo, x1=hi, y1=hi,line=dict(color="red", dash="dash"))
fig.update_layout(margin=dict(l=5, r=00, t=20, b=10)).show()

,PLTR,Market
PLTR,1.000000,0.528482
Market,0.528482,1.000000


# Beta
This is just covariance(stock, market) / (σ² of market returns).

β = 1: `1% move in market ==> 1% move in PLTR`

β < 1: `PLTR less sensitive.`

In [7]:
cov_matrix = comparison.cov()
σσ, cov = cov_matrix["Market"]["Market"], cov_matrix["PLTR"]["Market"]
β = cov / σσ
print(f"PLTR beta: {β:.2f}")

PLTR beta: 2.30


A beta of 2.30 describes an estimated slope of roughly 2.30 percentage points
of PLTR return per percentage point of SPY return.

# Maximum drawdown

In [8]:
percentage_from_peak = PLTR["Close"] / PLTR["Close"].cummax()
i = percentage_from_peak.idxmin()
print(f"Maximum drawdown: {percentage_from_peak.loc[i] - 1:.1%}, {i:%Y-%m-%d}")
print(f"Peak: {PLTR["Close"].cummax().loc[i]:.2f}")
print(f"Bottom: {PLTR["Close"].loc[i]:.2f}")

Maximum drawdown: -40.6%, 2025-04-04
Peak: 124.62
Bottom: 74.01


## Conclusions

PLTR showed greater daily-return variability than SPY. Its returns were
positively correlated with the benchmark, with an estimated beta of 2.30.

The largest observed drawdown was approximately 40.6%. These findings describe
this historical window and provide context for the simulation assumptions we will use later.